# NHL Contextual Special Teams Rating
**A replacement framework for raw PP% / PK%**

### What this builds
Instead of binary success/failure per opportunity, we compute:
- **Goals For/Against per 60 minutes of PP/PK time** — eliminates the short-opportunity distortion
- **Shots & scoring chances per 60** — process quality, not just outcomes
- **Score-state buckets** — Chasing (down 2+), Trailing (down 1), Neutral (tied), Leading (up 1), Protecting (up 2+)
- A composite **STR (Special Teams Rating)** per team per season

### Data source
NHL public API: `https://api-web.nhle.com/v1/`  
Play-by-play endpoint: `/v1/gamecenter/{game_id}/play-by-play`

### What we extract from PBP
- `penalty` events → who, when, duration, score at time of penalty
- `shot-on-goal`, `missed-shot`, `blocked-shot`, `goal` events → tagged by game strength situation
- Score state reconstructed from running goal tally at each event timestamp

In [ ]:
# ── Dependencies ──────────────────────────────────────────────────────────────
import requests
import pandas as pd
import numpy as np
import time
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)

BASE_URL = 'https://api-web.nhle.com/v1'
print('Libraries loaded.')

## 1. Configuration

In [ ]:
# ── Configure your run ────────────────────────────────────────────────────────
# Season format: YYYYYYYY  e.g. 20242025
SEASON        = '20242025'
GAME_TYPE     = 2          # 2 = regular season, 3 = playoffs

# To keep the initial run fast, cap the number of games fetched.
# Set to None to pull the full season (warning: ~1300 games, takes ~30 min).
MAX_GAMES     = 100        # start small, increase when comfortable

SLEEP_BETWEEN_REQUESTS = 0.25   # seconds — be polite to the NHL API

# Score-state bucket definitions  (from the PP team's perspective)
# goal_diff = pp_team_goals - pk_team_goals  at the time the penalty is called
def score_state_bucket(goal_diff: int) -> str:
    if   goal_diff <=  -2: return 'Chasing'
    elif goal_diff ==  -1: return 'Trailing'
    elif goal_diff ==   0: return 'Neutral'
    elif goal_diff ==   1: return 'Leading'
    else:                  return 'Protecting'

BUCKET_ORDER = ['Chasing', 'Trailing', 'Neutral', 'Leading', 'Protecting']
print(f'Config: season={SEASON}, game_type={GAME_TYPE}, max_games={MAX_GAMES}')

## 2. Fetch game IDs for the season

In [ ]:
def get_game_ids(season: str, game_type: int, max_games=None) -> list:
    """Pull all completed regular-season game IDs via the schedule endpoint."""
    # The schedule/season endpoint returns every game in the season
    url = f'{BASE_URL}/schedule/season/{season}/{game_type}'
    resp = requests.get(url)
    if resp.status_code != 200:
        # Fallback: build IDs manually (season has ~1312 regular-season games)
        print(f'Schedule endpoint returned {resp.status_code}, falling back to date-range walk.')
        return _walk_schedule_by_date(season, game_type, max_games)

    data = resp.json()
    game_ids = []

    # Response structure: {gameWeek: [{date, games: [{id, gameState, ...}]}]}
    for week in data.get('gameWeek', []):
        for game in week.get('games', []):
            if game.get('gameState') in ('OFF', 'FINAL'):   # completed games only
                game_ids.append(game['id'])

    if max_games:
        game_ids = game_ids[:max_games]

    print(f'Found {len(game_ids)} completed games for season {season}.')
    return game_ids


def _walk_schedule_by_date(season: str, game_type: int, max_games=None) -> list:
    """Fallback: walk week-by-week through the season to collect game IDs."""
    from datetime import date, timedelta
    year_start = int(season[:4])
    start_date = date(year_start, 10, 1)
    end_date   = date(year_start + 1, 6, 30)

    game_ids = []
    current  = start_date
    while current <= end_date:
        url  = f'{BASE_URL}/schedule/{current.isoformat()}'
        resp = requests.get(url)
        if resp.status_code == 200:
            data = resp.json()
            for week in data.get('gameWeek', []):
                for game in week.get('games', []):
                    if (game.get('gameType') == game_type and
                            game.get('gameState') in ('OFF', 'FINAL')):
                        game_ids.append(game['id'])
        current += timedelta(weeks=1)
        time.sleep(SLEEP_BETWEEN_REQUESTS)
        if max_games and len(game_ids) >= max_games:
            break

    game_ids = list(dict.fromkeys(game_ids))   # dedupe, preserve order
    if max_games:
        game_ids = game_ids[:max_games]
    print(f'Walked schedule: found {len(game_ids)} completed games.')
    return game_ids


game_ids = get_game_ids(SEASON, GAME_TYPE, MAX_GAMES)

## 3. Parse play-by-play into special teams events

In [ ]:
# ── Helpers ───────────────────────────────────────────────────────────────────

def period_seconds(period: int, time_in_period: str) -> int:
    """Convert period + MM:SS string to total seconds from puck drop."""
    m, s = map(int, time_in_period.split(':'))
    return (period - 1) * 1200 + m * 60 + s   # 1200 s per regulation period


def parse_game(game_id: int) -> pd.DataFrame | None:
    """
    Fetch one game's PBP and return a DataFrame of special-teams time slices.
    Each row = one second of PP/PK time, tagged with:
        game_id, pp_team, pk_team, score_bucket,
        goals_for, goals_against, shots_for, shots_against,
        missed_shots_for, blocked_shots_for
    """
    url  = f'{BASE_URL}/gamecenter/{game_id}/play-by-play'
    resp = requests.get(url)
    if resp.status_code != 200:
        return None
    data = resp.json()

    home_team = data['homeTeam']['abbrev']
    away_team = data['awayTeam']['abbrev']

    plays = data.get('plays', [])

    # ── Pass 1: build a running score dict and collect events ─────────────────
    score   = {home_team: 0, away_team: 0}     # live score tracker
    events  = []                                # all events with timestamps

    for play in plays:
        etype = play.get('typeDescKey', '')
        period = play.get('periodDescriptor', {}).get('number', 0)
        time_str = play.get('timeInPeriod', '0:00')
        abs_sec = period_seconds(period, time_str)

        details   = play.get('details', {}) or {}
        situation = play.get('situationCode', '')   # e.g. '1551' = home 5v4

        # Track goals to maintain live score
        if etype == 'goal':
            scoring_team = details.get('eventOwnerTeamId')
            # Map team id → abbrev
            if scoring_team == data['homeTeam']['id']:
                score[home_team] += 1
            elif scoring_team == data['awayTeam']['id']:
                score[away_team] += 1

        events.append({
            'abs_sec'  : abs_sec,
            'period'   : period,
            'etype'    : etype,
            'situation': situation,
            'details'  : details,
            'score_home': score[home_team],
            'score_away': score[away_team],
            'team_id'  : details.get('eventOwnerTeamId'),
        })

    # ── Pass 2: identify penalty windows ─────────────────────────────────────
    # We look for penalty events and record [start_sec, end_sec, pp_team, pk_team]
    # Penalties can overlap (5v3 etc.) but we handle each separately.

    penalty_windows = []

    for play in plays:
        if play.get('typeDescKey') != 'penalty':
            continue

        period   = play.get('periodDescriptor', {}).get('number', 0)
        time_str = play.get('timeInPeriod', '0:00')
        start    = period_seconds(period, time_str)
        details  = play.get('details', {}) or {}

        duration_min = details.get('duration', 2)         # minutes
        penalized_id = details.get('committedByPlayerId')  # player
        pen_team_id  = details.get('eventOwnerTeamId')

        if pen_team_id is None:
            continue

        # Identify pk_team abbrev and pp_team abbrev
        if pen_team_id == data['homeTeam']['id']:
            pk_team = home_team
            pp_team = away_team
        else:
            pk_team = away_team
            pp_team = home_team

        end = start + duration_min * 60

        # Score at time of penalty
        # Find the most recent score state in events up to this second
        score_home_at_pen = 0
        score_away_at_pen = 0
        for ev in events:
            if ev['abs_sec'] <= start:
                score_home_at_pen = ev['score_home']
                score_away_at_pen = ev['score_away']
            else:
                break

        if pp_team == home_team:
            goal_diff = score_home_at_pen - score_away_at_pen
        else:
            goal_diff = score_away_at_pen - score_home_at_pen

        penalty_windows.append({
            'game_id'   : game_id,
            'pp_team'   : pp_team,
            'pk_team'   : pk_team,
            'start_sec' : start,
            'end_sec'   : end,
            'goal_diff' : goal_diff,
            'bucket'    : score_state_bucket(goal_diff),
        })

    if not penalty_windows:
        return None

    pw_df = pd.DataFrame(penalty_windows)

    # ── Pass 3: tag shot/goal events to an open penalty window ───────────────
    shot_types = {'shot-on-goal', 'goal', 'missed-shot', 'blocked-shot'}

    rows = []
    for _, pen in pw_df.iterrows():
        pp_team  = pen['pp_team']
        pk_team  = pen['pk_team']
        start    = pen['start_sec']
        end      = pen['end_sec']
        bucket   = pen['bucket']

        # Find the actual end: did a goal or counter-penalty shorten this PP?
        actual_end = end
        for ev in events:
            if ev['abs_sec'] < start:
                continue
            if ev['abs_sec'] > end:
                break
            # PP-ending goal: a goal scored by pp_team ends the minor penalty
            if (ev['etype'] == 'goal' and
                    ev['team_id'] == (data['homeTeam']['id'] if pp_team == home_team
                                       else data['awayTeam']['id'])):
                actual_end = ev['abs_sec']
                break

        actual_duration = max(actual_end - start, 1)   # at least 1 second

        pp_goals  = pp_shots  = pp_missed = pp_blocked = 0
        pk_goals  = pk_shots  = 0

        pp_team_id = data['homeTeam']['id'] if pp_team == home_team else data['awayTeam']['id']
        pk_team_id = data['homeTeam']['id'] if pk_team == home_team else data['awayTeam']['id']

        for ev in events:
            if ev['abs_sec'] < start or ev['abs_sec'] > actual_end:
                continue
            if ev['etype'] not in shot_types:
                continue
            team = ev['team_id']
            etype = ev['etype']
            if team == pp_team_id:
                if   etype == 'goal':           pp_goals   += 1
                elif etype == 'shot-on-goal':   pp_shots   += 1
                elif etype == 'missed-shot':    pp_missed  += 1
                elif etype == 'blocked-shot':   pp_blocked += 1
            elif team == pk_team_id:
                if   etype == 'goal':           pk_goals   += 1
                elif etype == 'shot-on-goal':   pk_shots   += 1

        rows.append({
            'game_id'          : game_id,
            'pp_team'          : pp_team,
            'pk_team'          : pk_team,
            'bucket'           : bucket,
            'goal_diff_at_pen' : pen['goal_diff'],
            'pp_duration_sec'  : actual_duration,
            'pp_goals'         : pp_goals,
            'pp_shots_on_goal' : pp_shots,
            'pp_missed_shots'  : pp_missed,
            'pp_blocked_shots' : pp_blocked,
            'pk_goals_against' : pk_goals,    # shorthanded goals
            'pk_shots_against' : pk_shots,
        })

    return pd.DataFrame(rows) if rows else None


print('Parser defined.')

## 4. Pull data for all games

In [ ]:
all_frames = []
errors     = []

for gid in tqdm(game_ids, desc='Fetching PBP'):
    try:
        df = parse_game(gid)
        if df is not None and not df.empty:
            all_frames.append(df)
    except Exception as e:
        errors.append((gid, str(e)))
    time.sleep(SLEEP_BETWEEN_REQUESTS)

print(f'\nParsed {len(all_frames)} games successfully. {len(errors)} errors.')
if errors:
    print('Error game IDs:', [e[0] for e in errors[:5]], '...')

In [ ]:
raw_df = pd.concat(all_frames, ignore_index=True)
print(f'Raw event table: {raw_df.shape[0]:,} penalty windows across {raw_df["game_id"].nunique()} games.')
raw_df.head()

## 5. Aggregate to team-level special teams metrics

In [ ]:
# ── Power Play aggregation ────────────────────────────────────────────────────
pp_agg = (
    raw_df
    .groupby(['pp_team', 'bucket'])
    .agg(
        opportunities   = ('game_id',          'count'),
        total_sec       = ('pp_duration_sec',   'sum'),
        goals           = ('pp_goals',          'sum'),
        shots_on_goal   = ('pp_shots_on_goal',  'sum'),
        missed_shots    = ('pp_missed_shots',   'sum'),
        blocked_shots   = ('pp_blocked_shots',  'sum'),
    )
    .reset_index()
    .rename(columns={'pp_team': 'team'})
)

pp_agg['total_min']         = pp_agg['total_sec'] / 60
pp_agg['goals_per_60']      = pp_agg['goals']         / pp_agg['total_min'] * 60
pp_agg['shots_per_60']      = pp_agg['shots_on_goal'] / pp_agg['total_min'] * 60
pp_agg['shot_attempts_per_60'] = (pp_agg['shots_on_goal'] +
                                   pp_agg['missed_shots'] +
                                   pp_agg['blocked_shots']) / pp_agg['total_min'] * 60
pp_agg['shooting_pct']      = np.where(
    pp_agg['shots_on_goal'] > 0,
    pp_agg['goals'] / pp_agg['shots_on_goal'] * 100,
    np.nan
)
pp_agg['type'] = 'PP'

# ── Penalty Kill aggregation ──────────────────────────────────────────────────
pk_agg = (
    raw_df
    .groupby(['pk_team', 'bucket'])
    .agg(
        opportunities   = ('game_id',          'count'),
        total_sec       = ('pp_duration_sec',   'sum'),
        goals_against   = ('pp_goals',          'sum'),   # goals allowed on PK
        sh_goals        = ('pk_goals_against',  'sum'),   # shorthanded goals scored
        shots_against   = ('pk_shots_against',  'sum'),
    )
    .reset_index()
    .rename(columns={'pk_team': 'team'})
)

pk_agg['total_min']         = pk_agg['total_sec'] / 60
pk_agg['goals_against_per_60'] = pk_agg['goals_against'] / pk_agg['total_min'] * 60
pk_agg['sh_goals_per_60']   = pk_agg['sh_goals']         / pk_agg['total_min'] * 60
pk_agg['shots_against_per_60'] = pk_agg['shots_against'] / pk_agg['total_min'] * 60
pk_agg['type'] = 'PK'

print('PP aggregation:')
display(pp_agg.sort_values(['team','bucket']).head(20))
print('\nPK aggregation:')
display(pk_agg.sort_values(['team','bucket']).head(20))

## 6. Composite Special Teams Rating (STR)

In [ ]:
# ── Overall (all-bucket) composite ───────────────────────────────────────────
# Combine PP goals/60 and PK goals-against/60 into a single team rating.
# We z-score each component so they're on a common scale,
# then weight: PP goals/60 (40%), PP shot attempts/60 (10%),
#              PK goals against/60 (-40%, inverted), PK shots against/60 (-10%, inverted)

pp_overall = (
    raw_df
    .groupby('pp_team')
    .agg(
        pp_min         = ('pp_duration_sec', lambda x: x.sum() / 60),
        pp_goals       = ('pp_goals',        'sum'),
        pp_sog         = ('pp_shots_on_goal','sum'),
        pp_attempts    = ('pp_shots_on_goal','sum'),   # extend below
    )
    .reset_index()
    .rename(columns={'pp_team': 'team'})
)
# Add missed + blocked for total attempts
extra = raw_df.groupby('pp_team')[['pp_missed_shots','pp_blocked_shots']].sum().reset_index().rename(columns={'pp_team':'team'})
pp_overall = pp_overall.merge(extra, on='team')
pp_overall['pp_attempts'] = pp_overall['pp_sog'] + pp_overall['pp_missed_shots'] + pp_overall['pp_blocked_shots']
pp_overall['pp_goals_per_60']    = pp_overall['pp_goals']    / pp_overall['pp_min'] * 60
pp_overall['pp_attempts_per_60'] = pp_overall['pp_attempts'] / pp_overall['pp_min'] * 60

pk_overall = (
    raw_df
    .groupby('pk_team')
    .agg(
        pk_min         = ('pp_duration_sec', lambda x: x.sum() / 60),
        pk_ga          = ('pp_goals',        'sum'),
        pk_shots_ag    = ('pk_shots_against','sum'),
    )
    .reset_index()
    .rename(columns={'pk_team': 'team'})
)
pk_overall['pk_ga_per_60']       = pk_overall['pk_ga']       / pk_overall['pk_min'] * 60
pk_overall['pk_shots_ag_per_60'] = pk_overall['pk_shots_ag'] / pk_overall['pk_min'] * 60

str_df = pp_overall[['team','pp_goals_per_60','pp_attempts_per_60','pp_min']].merge(
         pk_overall[['team','pk_ga_per_60','pk_shots_ag_per_60','pk_min']], on='team')


def z_score(series):
    return (series - series.mean()) / series.std()

str_df['z_pp_goals']    =  z_score(str_df['pp_goals_per_60'])      # higher = better
str_df['z_pp_volume']   =  z_score(str_df['pp_attempts_per_60'])   # higher = better
str_df['z_pk_ga']       = -z_score(str_df['pk_ga_per_60'])         # lower ga = better → invert
str_df['z_pk_shots']    = -z_score(str_df['pk_shots_ag_per_60'])   # lower = better → invert

# Weighted composite
str_df['STR'] = (str_df['z_pp_goals']  * 0.40 +
                 str_df['z_pp_volume'] * 0.10 +
                 str_df['z_pk_ga']    * 0.40 +
                 str_df['z_pk_shots'] * 0.10)

str_df = str_df.sort_values('STR', ascending=False).reset_index(drop=True)
str_df.index += 1   # 1-based rank

print('=== Composite Special Teams Rating (STR) — All Score States ===')
display(str_df[['team','pp_goals_per_60','pp_attempts_per_60',
                'pk_ga_per_60','pk_shots_ag_per_60','STR']].round(3))

## 7. Score-bucket breakdown — PP goals/60 by team

In [ ]:
# Pivot: teams as rows, score buckets as columns
pp_pivot = (
    pp_agg[pp_agg['total_min'] >= 1]   # filter out tiny samples
    .pivot_table(index='team', columns='bucket', values='goals_per_60', aggfunc='first')
    .reindex(columns=BUCKET_ORDER)
    .round(2)
)

pk_pivot = (
    pk_agg[pk_agg['total_min'] >= 1]
    .pivot_table(index='team', columns='bucket', values='goals_against_per_60', aggfunc='first')
    .reindex(columns=BUCKET_ORDER)
    .round(2)
)

print('=== PP Goals per 60 by Score State ===')
display(pp_pivot)
print('\n=== PK Goals Against per 60 by Score State ===')
display(pk_pivot)

## 8. Visualizations

In [ ]:
# ── Chart 1: Top 15 teams by STR ──────────────────────────────────────────────
top_n    = min(15, len(str_df))
plot_df  = str_df.head(top_n)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#1a78cf' if v >= 0 else '#d62728' for v in plot_df['STR']]
bars = ax.barh(plot_df['team'][::-1], plot_df['STR'][::-1], color=colors[::-1])
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Composite STR (z-score weighted)')
ax.set_title(f'Special Teams Rating — Top {top_n} Teams\n{SEASON[:4]}-{SEASON[4:]} Season ({len(game_ids)} games sampled)')
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
plt.tight_layout()
plt.show()

In [ ]:
# ── Chart 2: PP goals/60 by score bucket — league average bar chart ───────────
league_pp = (
    raw_df
    .groupby('bucket')
    .agg(total_min=('pp_duration_sec', lambda x: x.sum()/60),
         total_goals=('pp_goals','sum'))
    .assign(goals_per_60=lambda d: d['total_goals'] / d['total_min'] * 60)
    .reindex(BUCKET_ORDER)
)

fig, ax = plt.subplots(figsize=(8, 5))
bar_colors = ['#d62728','#ff7f0e','#2ca02c','#1a78cf','#9467bd']
ax.bar(league_pp.index, league_pp['goals_per_60'], color=bar_colors)
ax.set_ylabel('PP Goals per 60 minutes')
ax.set_title(f'League-Average PP Goals/60 by Score State\n{SEASON[:4]}-{SEASON[4:]}')
ax.set_xlabel('Score State (from PP team perspective)')
for i, (idx, row) in enumerate(league_pp.iterrows()):
    ax.text(i, row['goals_per_60'] + 0.05, f"{row['goals_per_60']:.2f}", ha='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ── Chart 3: PP vs PK scatter — identify well-rounded vs specialist teams ────
fig, ax = plt.subplots(figsize=(9, 7))

scatter = ax.scatter(
    str_df['pp_goals_per_60'],
    str_df['pk_ga_per_60'],
    c=str_df['STR'], cmap='RdYlGn', s=80, edgecolors='gray', linewidths=0.4
)
plt.colorbar(scatter, ax=ax, label='STR')

# Quadrant lines at league means
ax.axvline(str_df['pp_goals_per_60'].mean(), color='gray', linestyle='--', linewidth=0.8)
ax.axhline(str_df['pk_ga_per_60'].mean(),   color='gray', linestyle='--', linewidth=0.8)

for _, row in str_df.iterrows():
    ax.annotate(row['team'], (row['pp_goals_per_60'], row['pk_ga_per_60']),
                fontsize=7, ha='left', va='bottom',
                xytext=(3, 3), textcoords='offset points')

ax.set_xlabel('PP Goals per 60 min  (higher = better PP)')
ax.set_ylabel('PK Goals Against per 60 min  (lower = better PK)')
ax.set_title(f'PP Efficiency vs PK Efficiency — All Score States\n{SEASON[:4]}-{SEASON[4:]}')

# Quadrant labels
xlim = ax.get_xlim(); ylim = ax.get_ylim()
ax.text(xlim[1]*0.97, ylim[0]*1.01, 'Elite PP\nStruggling PK', ha='right', fontsize=7, color='gray')
ax.text(xlim[0]*1.01, ylim[0]*1.01, 'Elite both',               ha='left',  fontsize=7, color='gray')

plt.tight_layout()
plt.show()

## 9. Team deep-dive — score-bucket breakdown

In [ ]:
# ── Change TEAM_TO_INSPECT to any team abbreviation in your dataset ───────────
TEAM_TO_INSPECT = str_df.iloc[0]['team']   # defaults to top STR team

print(f'\n=== Deep Dive: {TEAM_TO_INSPECT} ===')

pp_team_df = pp_agg[pp_agg['team'] == TEAM_TO_INSPECT].set_index('bucket').reindex(BUCKET_ORDER)
pk_team_df = pk_agg[pk_agg['team'] == TEAM_TO_INSPECT].set_index('bucket').reindex(BUCKET_ORDER)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# PP
axes[0].bar(pp_team_df.index, pp_team_df['goals_per_60'], color='#1a78cf')
axes[0].set_title(f'{TEAM_TO_INSPECT} — PP Goals/60 by Score State')
axes[0].set_ylabel('Goals per 60 PP minutes')
axes[0].set_xlabel('Score State')
for i, (idx, row) in enumerate(pp_team_df.iterrows()):
    if not np.isnan(row['goals_per_60']):
        axes[0].text(i, row['goals_per_60'] + 0.02,
                     f"{row['goals_per_60']:.2f}\n({row['total_min']:.0f}m)",
                     ha='center', fontsize=8)

# PK
axes[1].bar(pk_team_df.index, pk_team_df['goals_against_per_60'], color='#d62728')
axes[1].set_title(f'{TEAM_TO_INSPECT} — PK Goals Against/60 by Score State')
axes[1].set_ylabel('Goals Against per 60 PK minutes')
axes[1].set_xlabel('Score State')
for i, (idx, row) in enumerate(pk_team_df.iterrows()):
    if not np.isnan(row['goals_against_per_60']):
        axes[1].text(i, row['goals_against_per_60'] + 0.02,
                     f"{row['goals_against_per_60']:.2f}\n({row['total_min']:.0f}m)",
                     ha='center', fontsize=8)

plt.suptitle(f'{TEAM_TO_INSPECT} Special Teams — {SEASON[:4]}-{SEASON[4:]}', fontsize=13)
plt.tight_layout()
plt.show()

print(f'\nPP summary:')
display(pp_team_df[['opportunities','total_min','goals','shots_on_goal',
                     'goals_per_60','shots_per_60','shooting_pct']].round(2))
print(f'\nPK summary:')
display(pk_team_df[['opportunities','total_min','goals_against',
                     'sh_goals','goals_against_per_60','sh_goals_per_60']].round(2))

## 10. Export

In [ ]:
str_df.to_csv(f'nhl_str_{SEASON}.csv', index=False)
pp_agg.to_csv(f'nhl_pp_by_bucket_{SEASON}.csv', index=False)
pk_agg.to_csv(f'nhl_pk_by_bucket_{SEASON}.csv', index=False)
raw_df.to_csv(f'nhl_raw_penalty_windows_{SEASON}.csv', index=False)

print('Exported:')
print(f'  nhl_str_{SEASON}.csv              — composite ratings')
print(f'  nhl_pp_by_bucket_{SEASON}.csv     — PP metrics by score bucket')
print(f'  nhl_pk_by_bucket_{SEASON}.csv     — PK metrics by score bucket')
print(f'  nhl_raw_penalty_windows_{SEASON}.csv  — raw penalty window events')